# Portfolio analytics

This notebook only orchestrates: it calls `transactions` (trade logic), `prices` (Yahoo Finance fetch + cache), `returns` (CAGR / HYSA benchmark math), and `visualization` (all charts). No logic lives in this notebook itself — see `docs/architecture.md` for the module map.

In [ ]:
from datetime import date
from pathlib import Path

from trades import prices, returns, transactions, visualization
from trades.config import AggregationConfig, PriceApiConfig, ReturnsConfig

TRADES_CSV = Path("..") / "data" / "20260701_trades.csv"
AS_OF_DATE = date.today()  # change this to price the portfolio as of any past date

# Every tunable parameter lives on one of these config objects (see
# docs/architecture.md#configuration) — nothing here is a hidden default.
aggregation_config = AggregationConfig()
price_api_config = PriceApiConfig()
returns_config = ReturnsConfig()

## 1. Load, enrich, and aggregate trades

`load_raw_trades` validates every CSV row through the `RawTrade` model. `enrich_trades` adds `usd_per_share`. `aggregate_same_day_trades` then merges same-day, same-symbol fills executed within 0.01% of each other into one row (summed shares/USD, recomputed $/share) — this is the dataset used for everything downstream.

In [ ]:
raw = transactions.load_raw_trades(TRADES_CSV)
enriched = transactions.enrich_trades(raw)
trades = transactions.aggregate_same_day_trades(enriched, aggregation_config)
print(f"{len(raw)} raw fills -> {len(trades)} aggregated trades")
trades

## 2. Investment schedule

Totals per symbol, invested-per-month (overall and per symbol), the daily investment timeline (with gaps between buys), and a pie breakdown with a menu to switch between whole-portfolio-by-symbol and any one symbol's by-date split.

In [ ]:
total_by_symbol = transactions.total_invested_by_symbol(trades)
print("Total invested to date, by symbol:")
print(total_by_symbol.to_string())
print(f"\nTotal invested to date, whole portfolio: ${total_by_symbol.sum():,.2f}")

In [ ]:
monthly = transactions.monthly_invested(trades)
monthly

In [ ]:
visualization.plot_monthly_invested(monthly).show()

In [ ]:
daily = transactions.daily_investment_timeline(trades)
visualization.plot_daily_investment_timeline(daily).show()

In [ ]:
pie_options = transactions.pie_chart_options(trades)
visualization.plot_investment_pie(pie_options).show()

## 3. Price history

One local cache file per symbol (`data/prices/{SYMBOL}.csv`), each call only fetching the date range missing since the last run — see `docs/architecture.md` for the cache design. History goes back to the first trade date across the whole portfolio.

In [ ]:
symbols = sorted(trades["symbol"].unique())
first_trade_date = trades["trade_date"].min().date()

price_histories = prices.update_price_caches(
    symbols, since=first_trade_date, as_of=AS_OF_DATE, config=price_api_config
)
for symbol, history in price_histories.items():
    latest_close = history["close"].iloc[-1]
    print(f"{symbol}: {len(history)} trading days cached, latest close {latest_close:.2f}")

## 4. Returns vs. a HYSA benchmark

For each trade: current price, days held, total return, CAGR-style annualized return, the compounded HYSA return over the same window (rate set by `returns_config.hysa_annual_rate`, default 4%), and the resulting alpha. See `docs/returns.md` for the derivation of each step.

In [ ]:
def price_lookup(symbol: str, as_of: date) -> float | None:
    return prices.price_as_of(price_histories[symbol], as_of)


returns_df = returns.build_returns_table(
    trades, price_lookup, as_of=AS_OF_DATE, config=returns_config
)

display_table = returns_df[
    [
        "trade_date",
        "symbol",
        "usd_per_share",
        "current_price",
        "days_held",
        "total_return_pct",
        "annualized_return_pct",
        "hysa_period_return_pct",
        "alpha_period_pct",
    ]
].rename(columns={"usd_per_share": "price_paid"})
display_table

In [ ]:
numeric_cols = display_table.select_dtypes("number").columns
display_rounded = display_table.assign(**{c: display_table[c].round(2) for c in numeric_cols})
visualization.render_table(display_rounded, title=f"Returns as of {AS_OF_DATE}").show()

portfolio_alpha = returns.portfolio_alpha_pct(returns_df)
label = (
    f"Dollar-weighted portfolio alpha vs. {returns_config.hysa_annual_rate:.0%} HYSA "
    "(period, not annualized)"
)
print(f"{label}: {portfolio_alpha:+.2f}%")

## 5. Annualized return curve

Per-trade annualized return against days held, with a fitted trend and the flat HYSA benchmark line. Short holds annualize into large, noisy numbers by design — that's why the combined alpha above uses period alpha instead of this annualized figure.

In [ ]:
trend_x, trend_y = returns.fit_trend(
    returns_df["days_held"].to_numpy(),
    returns_df["annualized_return_pct"].to_numpy(),
    returns_config,
)
visualization.plot_return_curve(returns_df, trend_x, trend_y, returns_config).show()